# Survivor Machine — サイトの木は本物か? / Is the website tree real?

サイトの **Survivor Machine** で「自分は助かるか?」を遊んだあと、このノートで確かめます。

- サイトの木に埋め込んだ生存確率は、**本物の Titanic データから出した実際の割合**であることを確認する
- 本物の決定木(深さ3)を sklearn で学習し、**特徴量重要度**を出す → 「運命を変える最小の一手」がなぜ性別だったか分かる
- (任意) GAS で集めたクラスの予想を読み込み、**人間 vs モデル**を比べる

このノートは公開リポジトリ **github.com/itoksk/summer-ai-materials** の `materials/notebooks/` にあります。

## Step 1 — 本物のデータを読む / Load the real data

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
try:
    df = pd.read_csv(url)
except Exception:
    print("ダウンロードできない場合: 先生から titanic.csv をもらって左の folder にアップロードしてください")
    df = pd.read_csv("titanic.csv")
print("乗客:", len(df), "人  全体の生存率:", round(df["Survived"].mean(), 3))
df.head()

## Step 2 — サイトの数字 vs 実データ / Website numbers vs real data

サイトのウィジェットは、これらの確率を木に埋め込んでいます。本当にデータと一致する?

In [ ]:
g = df.dropna(subset=["Age"]).copy()

def rate(mask):
    sub = g[mask]
    return (len(sub), round(sub["Survived"].mean(), 2)) if len(sub) else (0, None)

print("                 サイトの値   実データ (人数, 生存率)")
print("女性 1等          0.97        ", rate((g.Sex=="female") & (g.Pclass==1)))
print("女性 2等          0.92        ", rate((g.Sex=="female") & (g.Pclass==2)))
print("女性 3等          0.50        ", rate((g.Sex=="female") & (g.Pclass==3)))
print("男児(<=6) 1-2等   0.83        ", rate((g.Sex=="male") & (g.Age<=6) & (g.Pclass<=2)))
print("男児(<=6) 3等     0.36        ", rate((g.Sex=="male") & (g.Age<=6) & (g.Pclass==3)))
print("男性(>6) 1等      0.36        ", rate((g.Sex=="male") & (g.Age>6) & (g.Pclass==1)))
print("男性(>6) 2-3等    0.13        ", rate((g.Sex=="male") & (g.Age>6) & (g.Pclass>=2)))

**観察 / Observe**: サイトの数字は作り物ではなく、110年前の実データの割合そのもの。だから「自分の判定」も本物だった。

## Step 3 — 本物の決定木と特徴量重要度 / The real tree + feature importance

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

d = df.dropna(subset=["Age"]).copy()
d["Sex_male"] = (d["Sex"] == "male").astype(int)
X = d[["Sex_male", "Pclass", "Age"]]
y = d["Survived"]
clf = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X, y)

plt.figure(figsize=(14, 7))
plot_tree(clf, feature_names=["Sex(male=1)", "Pclass", "Age"],
          class_names=["died", "survived"], filled=True, rounded=True)
plt.show()

print("特徴量重要度 / feature importance:")
for name, imp in zip(X.columns, clf.feature_importances_):
    print(" ", name, round(imp, 3))

**つながり / Connect**: 一番大きい重要度は **Sex**。だからウィジェットで「運命を変える最小の一手」が(多くの場合)性別だった。木はそれを誰にも教わらず、データだけから決めた。

## Step 4 — モデルが測っていないもの / What the model never measured

ウィジェットの「勇気・泳げる・運がいい」をいくらONにしても判定は動かなかった。理由はここにある —
**データにそんな列が無い**。モデルは列にあるものしか知らない。

In [ ]:
print("データにある列 / columns the model could ever see:")
print(list(df.columns))
print()
print("'勇気' 'やさしさ' 'あきらめない心' のような列は、どこにも無い。")

## Step 5 (任意) — 人間 vs モデル / Class guesses vs the model

GAS のスプレッドシートを「ファイル → ダウンロード → カンマ区切り(.csv)」で `responses.csv` にして、
左の folder アイコンからアップロードしてから実行(無ければ自動でスキップ)。

In [ ]:
try:
    cls = pd.read_csv("responses.csv")

    def model_predict(sex, pclass, age):
        if sex == "female":
            p = 0.97 if pclass == 1 else (0.92 if pclass == 2 else 0.5)
        elif age <= 6:
            p = 0.36 if pclass == 3 else 0.83
        else:
            p = 0.36 if pclass == 1 else 0.13
        return "survive" if p >= 0.5 else "dies"

    cls["model"] = cls.apply(
        lambda r: model_predict(r["sex"], int(r["pclass"]), float(r["age"])), axis=1)
    agree = (cls["guess"] == cls["model"]).mean()
    print("クラス人数:", len(cls))
    print("モデルが survive と判定:", int((cls["model"] == "survive").sum()))
    print("みんなが survive と予想:", int((cls["guess"] == "survive").sum()))
    print("人間とモデルの一致率:", round(agree * 100), "%")
except FileNotFoundError:
    print("responses.csv が無いのでスキップ。サイトのウィジェットだけでも体験は完結します。")

## 問い / Questions
- サイトの数字と実データはどれくらい一致した? ずれた所は、なぜ?
- 特徴量重要度で Sex が一番大きいのはなぜ? それは「世界の真実」か「1912年の救命ボートの真実」か?
- 人間の予想とモデル、どちらがよく当たった? モデルが見落としている人間的なものは何?